In [1]:
import json
import shutil
import time
from collections import Counter, defaultdict
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from ultralytics import YOLO

import costing as cs

print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())

torch 2.11.0+cpu | cuda: False


In [2]:
# switches
FAST = False        # True = a 1-epoch smoke test
SKIP_TRAIN = False  # True = reuse weights/best.pt and only redo evaluation and costing

# dataset and splits
WORKSPACE, PROJECT, VERSION = "cs614proj", "fire_smoke_detection-eskmo", 1
DATASET_DIR = Path("datasets") / f"{PROJECT}-v{VERSION}" 
SPLITS = ("train", "valid", "test")
POSITIVE_NAMES = {"fire", "smoke"}
AUGMENT_FACTOR = 3

# model and training parameters
BASE_MODEL = "yolo11n.pt"
if FAST:
    IMGSZ, EPOCHS, PATIENCE, BATCH, FRACTION = 320, 1, 1, 8, 0.1
else:
    IMGSZ, EPOCHS, PATIENCE, BATCH, FRACTION = 416, 25, 10, 16, 1.0
SEED = 0
DEVICE = 0 if torch.cuda.is_available() else "cpu"

# directories and files
WEIGHTS_DIR, RESULTS_DIR = Path("weights"), Path("results")
BEST_WEIGHTS = WEIGHTS_DIR / "best.pt"
WEIGHTS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

np.random.seed(SEED)
print(f"FAST={FAST}  SKIP_TRAIN={SKIP_TRAIN}  model={BASE_MODEL}  imgsz={IMGSZ}  "
      f"epochs={EPOCHS}  device={DEVICE}")

FAST=False  SKIP_TRAIN=False  model=yolo11n.pt  imgsz=416  epochs=25  device=cpu


In [3]:
from api_key import load_api_key

if not (DATASET_DIR / "data.yaml").exists():
    from roboflow import Roboflow

    key = load_api_key()
    key = key[0] if isinstance(key, (tuple, list)) else key
    rf = Roboflow(api_key=key)
    rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download(
        "yolov11", location=str(DATASET_DIR)
    )

cfg = yaml.safe_load(open(DATASET_DIR / "data.yaml"))
names = cfg["names"]
CLASS_NAMES = [names[i] for i in sorted(names)] if isinstance(names, dict) else list(names)
POS_IDS = {i for i, n in enumerate(CLASS_NAMES) if n.lower() in POSITIVE_NAMES}
assert len(POS_IDS) == len(POSITIVE_NAMES), f"expected {POSITIVE_NAMES} in {CLASS_NAMES}"


DATA_YAML = DATASET_DIR / "data_local.yaml"
yaml.safe_dump(
    {"path": str(DATASET_DIR.resolve()), "train": "train/images", "val": "valid/images",
     "test": "test/images", "names": CLASS_NAMES},
    open(DATA_YAML, "w"),
)
print("classes:", CLASS_NAMES, "| positive ids:", sorted(POS_IDS))

classes: ['None', 'fire', 'smoke'] | positive ids: [1, 2]


In [4]:
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def list_images(split):
    folder = DATASET_DIR / split / "images"
    return sorted(p for p in folder.iterdir() if p.suffix.lower() in IMG_EXT)


def read_labels(image_path):
    # [(class_id, x, y, w, h), ...]; a photo with no file, or an empty one, has no objects
    label = image_path.parent.parent / "labels" / f"{image_path.stem}.txt"
    rows = []
    if label.exists():
        for line in label.read_text().splitlines():
            parts = line.split()
            if len(parts) >= 5:
                rows.append((int(float(parts[0])), *map(float, parts[1:5])))
    return rows


def is_positive(image_path):
    return any(row[0] in POS_IDS for row in read_labels(image_path))


dataset_stats, table = {}, []
for split in SPLITS:
    images = list_images(split)
    instances, areas = Counter(), defaultdict(list)
    positive = only_none = empty = 0
    for path in images:
        labels = read_labels(path)
        ids = {row[0] for row in labels}
        for cls, _, _, w, h in labels:
            instances[CLASS_NAMES[cls]] += 1
            areas[CLASS_NAMES[cls]].append(w * h)
        if ids & POS_IDS:
            positive += 1
        elif ids:
            only_none += 1
        else:
            empty += 1
    entry = {
        "images": len(images),
        "positive_images": positive,
        "negative_images": only_none + empty,
        "negative_only_None_label": only_none,
        "negative_no_label": empty,
        "instances": dict(instances),
        "median_box_area_share": {c: float(np.median(a)) for c, a in areas.items()},
        "share_boxes_under_1pct_of_image": {
            c: float(np.mean(np.array(a) < 0.01)) for c, a in areas.items()
        },
    }
    if split == "train":
        entry["images_original_estimate"] = len(images) / AUGMENT_FACTOR
    dataset_stats[split] = entry
    row = {"split": split, "images": len(images), "positive": positive,
           "negative": only_none + empty}
    for c in CLASS_NAMES:
        row[f"n_{c}"] = instances.get(c, 0)
    for c in CLASS_NAMES:
        row[f"median_area_{c}"] = round(entry["median_box_area_share"].get(c, float("nan")), 4)
    table.append(row)

print(pd.DataFrame(table).to_string(index=False))
print(f"\ntrain is augmented {AUGMENT_FACTOR}x, so it holds about "
      f"{dataset_stats['train']['images'] / AUGMENT_FACTOR:.0f} original photos.")
print("valid and test are not augmented: those are real, distinct photos.")
json.dump(dataset_stats, open(RESULTS_DIR / "dataset_stats.json", "w"), indent=2)

split  images  positive  negative  n_None  n_fire  n_smoke  median_area_None  median_area_fire  median_area_smoke
train    1812       747      1065    1060    1380     1588               1.0            0.0792             0.0917
valid     301       117       184     184     199      209               1.0            0.0644             0.0906
 test     101        43        58      58      63       78               1.0            0.0776             0.0784

train is augmented 3x, so it holds about 604 original photos.
valid and test are not augmented: those are real, distinct photos.


In [ ]:
started = time.perf_counter()
if SKIP_TRAIN and BEST_WEIGHTS.exists():
    print("SKIP_TRAIN is True and weights/best.pt exists: not training.")
    trained_now = False
else:
    model = YOLO(BASE_MODEL)
    model.train(
        data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, patience=PATIENCE,
        seed=SEED, device=DEVICE, fraction=FRACTION, project="runs", name="findfire",
        exist_ok=True, plots=True,
    )
    run_dir = Path(model.trainer.save_dir)
    best = run_dir / "weights" / "best.pt"
    shutil.copy(best if best.exists() else run_dir / "weights" / "last.pt", BEST_WEIGHTS)
    if (run_dir / "results.csv").exists():
        shutil.copy(run_dir / "results.csv", RESULTS_DIR / "train_log.csv")
    trained_now = True

import ultralytics

train_settings = {
    "trained_now": trained_now,
    "fast_mode": FAST,
    "base_model": BASE_MODEL,
    "epochs_requested": EPOCHS,
    "patience": PATIENCE,
    "imgsz": IMGSZ,
    "batch": BATCH,
    "fraction_of_train_used": FRACTION,
    "seed": SEED,
    "device": "cuda" if DEVICE == 0 else "cpu",
    "ultralytics": ultralytics.__version__,
    "torch": torch.__version__,
    "training_seconds": round(time.perf_counter() - started, 1),
    "weights": str(BEST_WEIGHTS),
}
if trained_now or not (RESULTS_DIR / "train_settings.json").exists():
    json.dump(train_settings, open(RESULTS_DIR / "train_settings.json", "w"), indent=2)
else:  # keep the record of the real training run, not of this re-evaluation
    train_settings = json.load(open(RESULTS_DIR / "train_settings.json"))
    IMGSZ = train_settings["imgsz"]
print(json.dumps(train_settings, indent=2))

New https://pypi.org/project/ultralytics/8.4.189 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.173  Python-3.12.14 torch-2.11.0+cpu CPU (AMD Ryzen 7 7735HS with Radeon Graphics)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=datasets\fire_smoke_detection-eskmo-v1\data_local.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.